# Demo: Bias-Corrected Turnbull-Wakeman Approximation

This notebook demonstrates the main workflow of the project:

1. Price one arithmetic Asian option using the control variate Monte Carlo benchmark.
2. Price the same option using the Turnbull-Wakeman approximation.
3. Compute the Turnbull-Wakeman residual.
4. Train a simple polynomial Ridge residual model from the generated dataset.
5. Apply the bias correction to obtain a corrected Turnbull-Wakeman price.

The goal is to show how the project combines:

\[
\text{Analytical Approximation}
+
\text{Control Variate Benchmark}
+
\text{Residual Learning}.
\]

## 1. Import Packages and Project Functions

In [ ]:
import numpy as np
import pandas as pd

from control_variate import arithmetic_asian_cv
from approximation import turnbull_wakeman_arithmetic_asian_price

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Price One Arithmetic Asian Option

In [ ]:
# Option parameters
S0 = 100
K = 100
r = 0.05
sigma = 0.2
T = 1.0
n = 12

n_paths = 100_000
seed = 42
option_type = "call"

In [ ]:
# Control variate Monte Carlo benchmark
cv_result = arithmetic_asian_cv(
    S0=S0,
    K=K,
    r=r,
    sigma=sigma,
    T=T,
    n=n,
    n_paths=n_paths,
    seed=seed,
    option_type=option_type,
)

# Turnbull-Wakeman approximation
tw_price = turnbull_wakeman_arithmetic_asian_price(
    S0=S0,
    K=K,
    r=r,
    sigma=sigma,
    T=T,
    n=n,
    option_type=option_type,
)

tw_residual = tw_price - cv_result.price
scaled_residual = tw_residual / S0

In [ ]:
print("Control Variate Benchmark:", cv_result.price)
print("Control Variate Std. Error:", cv_result.std_error)
print("Turnbull-Wakeman Price:", tw_price)
print("TW Residual:", tw_residual)
print("Scaled TW Residual:", scaled_residual)
print("Variance Reduction:", cv_result.variance_reduction)

The residual is defined as:

$$
\varepsilon_{\mathrm{TW}}
=
C_{\mathrm{TW}} - V_{\mathrm{CV}}.
$$

A positive residual means that Turnbull-Wakeman overprices the option relative to the control variate benchmark.  
A negative residual means that Turnbull-Wakeman underprices the option.

## 3. Load the Residual Dataset

In [ ]:
dataset_path = "tw_residual_dataset_s0_grid.csv"

df = pd.read_csv(dataset_path)

df.head()

In [ ]:
print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

The dataset should contain option parameters, benchmark prices, Turnbull-Wakeman prices, residuals, and engineered features.

The target variable is the scaled residual:

$$
y = \frac{C_{\mathrm{TW}} - V_{\mathrm{CV}}}{S_0}.
$$

## 4. Construct Features and Target

In [ ]:
# If these feature columns already exist in the dataset, use them directly.
# Otherwise, construct them from the raw option parameters.

df = df.copy()

if "moneyness" not in df.columns:
    df["moneyness"] = df["K"] / df["S0"]

if "log_moneyness" not in df.columns:
    df["log_moneyness"] = np.log(df["K"] / df["S0"])

if "inv_n" not in df.columns:
    df["inv_n"] = 1.0 / df["n"]

if "inv_sqrt_n" not in df.columns:
    df["inv_sqrt_n"] = 1.0 / np.sqrt(df["n"])

if "scaled_residual" not in df.columns:
    # This assumes the dataset contains tw_residual.
    df["scaled_residual"] = df["tw_residual"] / df["S0"]

In [ ]:
feature_cols = [
    "log_moneyness",
    "sigma",
    "T",
    "inv_n",
    "inv_sqrt_n",
]

target_col = "scaled_residual"

X = df[feature_cols]
y = df[target_col]

X.head()

## 5. Train a Polynomial Ridge Residual Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

model = Pipeline(
    steps=[
        ("poly", PolynomialFeatures(degree=3, include_bias=False)),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1.0)),
    ]
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("Scaled residual model performance")
print("MAE:", mae)
print("RMSE:", rmse)
print("R^2:", r2)

This model is not learning the whole option price directly.  
It only learns the remaining structured residual of the Turnbull-Wakeman approximation:

$$
\widehat{y}
=
f_{\theta}(x).
$$

The unscaled residual estimate is:

$$
\widehat{\varepsilon}_{\mathrm{TW}}
=
S_0 \widehat{y}.
$$

## 6. Apply Bias Correction to One Option

In [ ]:
# Construct a one-row feature DataFrame for the option priced above.

x_new = pd.DataFrame(
    {
        "log_moneyness": [np.log(K / S0)],
        "sigma": [sigma],
        "T": [T],
        "inv_n": [1.0 / n],
        "inv_sqrt_n": [1.0 / np.sqrt(n)],
    }
)

predicted_scaled_residual = model.predict(x_new)[0]
predicted_residual = S0 * predicted_scaled_residual

corrected_price = tw_price - predicted_residual

print("Original TW Price:", tw_price)
print("Predicted Scaled Residual:", predicted_scaled_residual)
print("Predicted Residual:", predicted_residual)
print("Corrected TW Price:", corrected_price)
print("CV Benchmark:", cv_result.price)

This model does not learn the full option price directly.
Instead, it learns the remaining structured residual of the Turnbull-Wakeman approximation:

$$
\widehat{y}
=
f_{\theta}(x).
$$

The predicted scaled residual is then converted back to the original price scale:

$$
\widehat{\varepsilon}_{\mathrm{TW}}
=
S_0 \widehat{y}.
$$

## 7. Compare Original TW Error and Corrected TW Error

In [ ]:
original_error = tw_price - cv_result.price
corrected_error = corrected_price - cv_result.price

print("Original TW Error:", original_error)
print("Corrected TW Error:", corrected_error)
print("Absolute Original Error:", abs(original_error))
print("Absolute Corrected Error:", abs(corrected_error))

if abs(corrected_error) < abs(original_error):
    print("Bias correction improved the price for this example.")
else:
    print("Bias correction did not improve this example.")

## 8. Apply Correction to the Test Set

In [ ]:
test_df = df.loc[X_test.index].copy()

test_df["predicted_scaled_residual"] = y_pred
test_df["predicted_residual"] = test_df["S0"] * test_df["predicted_scaled_residual"]

# The dataset column names may differ depending on your script.
# Adjust these names if needed.
if "tw_price" in test_df.columns:
    tw_col = "tw_price"
elif "C_TW" in test_df.columns:
    tw_col = "C_TW"
else:
    raise ValueError("Cannot find TW price column. Expected 'tw_price' or 'C_TW'.")

if "cv_mc_price" in test_df.columns:
    cv_col = "cv_mc_price"
elif "V_CV" in test_df.columns:
    cv_col = "V_CV"
else:
    raise ValueError("Cannot find CV benchmark column. Expected 'cv_mc_price' or 'V_CV'.")

test_df["corrected_price"] = test_df[tw_col] - test_df["predicted_residual"]

test_df["original_error"] = test_df[tw_col] - test_df[cv_col]
test_df["corrected_error"] = test_df["corrected_price"] - test_df[cv_col]

test_df["original_abs_error"] = test_df["original_error"].abs()
test_df["corrected_abs_error"] = test_df["corrected_error"].abs()

test_df.head()

In [ ]:
original_mae = test_df["original_abs_error"].mean()
corrected_mae = test_df["corrected_abs_error"].mean()

original_rmse = (test_df["original_error"] ** 2).mean() ** 0.5
corrected_rmse = (test_df["corrected_error"] ** 2).mean() ** 0.5

improved_fraction = (
    test_df["corrected_abs_error"] < test_df["original_abs_error"]
).mean()

print("Original TW MAE:", original_mae)
print("Corrected TW MAE:", corrected_mae)
print("MAE Reduction:", 1 - corrected_mae / original_mae)

print("Original TW RMSE:", original_rmse)
print("Corrected TW RMSE:", corrected_rmse)
print("RMSE Reduction:", 1 - corrected_rmse / original_rmse)

print("Improved Fraction:", improved_fraction)

## 9. Save Demo Results

In [ ]:
output_path = "demo_bias_correction_results.csv"

test_df.to_csv(output_path, index=False)

print("Saved demo results to:", output_path)

## 10. Interpretation

This demo shows the core idea of the project:

1. The Turnbull-Wakeman approximation gives a fast analytical price.
2. The control variate Monte Carlo price is used as a high-accuracy benchmark.
3. The residual between TW and CV is structured rather than random.
4. A polynomial Ridge model can learn this residual surface.
5. The corrected price keeps the speed of TW while improving accuracy within the calibrated parameter grid.

The correction should be interpreted mainly as an interpolation-based improvement.  
Performance may be weaker when extrapolating outside the training grid, especially for extreme moneyness or long maturity regimes.